In [ ]:
import pandas as pd
import os

# Ensure the output directory exists
output_dir = './Output_Data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 1. Load the reports

df1 = pd.read_excel(r'C:\Users\USER\Downloads\Emarath_global\Data\agent_detaild_report.xlsx', 
                    sheet_name='Sheet1', skiprows=4, dtype={'MOBILE': str})

df2 = pd.read_excel(r'C:\Users\USER\Downloads\Emarath_global\Data\agent_detaild_report_2.xlsx', 
                    sheet_name='Sheet1', skiprows=4, dtype={'MOBILE': str})

df1['MOBILE'] = df1['MOBILE'].str.strip()
df2['MOBILE'] = df2['MOBILE'].str.strip()


df1['Is_In_Report_2'] = df1['MOBILE'].isin(df2['MOBILE'])

matches = df1[df1['Is_In_Report_2'] == True].copy()

print(f"Total rows in Report 1: {len(df1)}")
print(f"Total matches found in Report 2: {len(matches)}")


matches.to_excel(os.path.join(output_dir, 'matched_mobile_numbers.xlsx'), index=False)

matches.head()

Total rows in Report 1: 437
Total matches found in Report 2: 419


,AGENT_NAME,CUSTOMER_NAME,MOBILE,EMAIL,STATUS,CONTACT_SOURCE,SOURCE_ID,SOURCE_URL,CREATE_DATE,Is_In_Report_2
0,NaN,Anudin kk,919495486991,NaN,OPEN,ORGANIC,NaN,NaN,NaN,True
1,NaN,Vasu,966509856130,NaN,OPEN,ad,6.947847e+12,https://fb.me/47HP5j54p,NaN,True
2,NaN,Riyadh,966548610869,NaN,OPEN,ORGANIC,NaN,NaN,NaN,True
3,NaN,Mujeeb Pk,966554679603,NaN,OPEN,ORGANIC,NaN,NaN,NaN,True
5,ADHIL C K,mhmdalshhatmstfym,201120765600,NaN,OPEN,ORGANIC,NaN,NaN,NaN,True


In [ ]:
import pandas as pd
import os

# 1. Load the reports
file1 = r'C:\Users\USER\Downloads\Emarath_global\Data\agent_wise_lead_data.xlsx'
file2 = r'C:\Users\USER\Downloads\Emarath_global\Data\flowbee_detaild_report.xlsx'

# Loading data with string type for MOBILE to avoid data corruption
df1 = pd.read_excel(file1, sheet_name='Sheet1', skiprows=4, dtype={'MOBILE': str})
df2 = pd.read_excel(file2, sheet_name='Sheet1', skiprows=4, dtype={'MOBILE': str})

# 2. Pre-processing / Cleaning

df1['MOBILE'] = df1['MOBILE'].astype(str).str.strip()
df1['AGENT_NAME'] = df1['AGENT_NAME'].astype(str).str.strip().str.upper()

df2['MOBILE'] = df2['MOBILE'].astype(str).str.strip()
df2['AGENT_NAME'] = df2['AGENT_NAME'].astype(str).str.strip().str.upper()

# 3. Merge reports on MOBILE to compare Agents
comparison = pd.merge(
    df1, 
    df2[['MOBILE', 'AGENT_NAME']], 
    on='MOBILE', 
    how='left', 
    suffixes=('_Report1', '_Report2')
)


mismatch_filter = (comparison['AGENT_NAME_Report2'].isna()) | \
                  (comparison['AGENT_NAME_Report1'] != comparison['AGENT_NAME_Report2'])

mismatched_data = comparison[mismatch_filter].copy()

# 5. Clean up the output
mismatched_data['AGENT_NAME_Report2'] = mismatched_data['AGENT_NAME_Report2'].fillna('NOT IN REPORT 2')

# Sort by Agent Name from the first report
mismatched_data = mismatched_data.sort_values(by=['AGENT_NAME_Report1', 'MOBILE'])

# 6. Save and Display
output_file = './Output_Data/agent_mismatch_report.xlsx'
os.makedirs('./Output_Data', exist_ok=True)
mismatched_data.to_excel(output_file, index=False)

print(f"Mismatch Report Generated: {len(mismatched_data)} rows found.")
mismatched_data[['AGENT_NAME_Report1', 'MOBILE', 'AGENT_NAME_Report2']].head(10)

Mismatch Report Generated: 32 rows found.


,AGENT_NAME_Report1,MOBILE,AGENT_NAME_Report2
29,AJIN V,919717892683,NAJA TK
43,AJIN V,966554679603,NOT IN REPORT 2
78,ARUN BABU,8801798760941,MOHAMMED RAHIB K E
80,ARUN BABU,917845279730,SAFWAN K P
89,ARUN BABU,966504181442,SHAMNA N
96,ARUN BABU,966555306324,BURHANA N R
99,ARUN BABU,966564166035,AJIN V
100,ARUN BABU,966568854813,AJIN V
101,ARUN BABU,966570993549,AJIN V
111,ARUN BABU,971503720966,NOT IN REPORT 2


In [11]:
from gspread_pandas import Spread, conf
list_sheet = "https://docs.google.com/spreadsheets/d/1ZexyNYrwg1p7EG6q2S1Fv4XfliDKkWR0T1YeYXo84gU/edit?gid=362506180#gid=362506180"

config_path = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
# 2. Load the config from your service account JSON
c = conf.get_config(file_name=config_path)

# 3. Connect using that config object
spread = Spread(list_sheet, config=c)

In [12]:
try:
    # Load all data
    df_sheet = spread.sheet_to_df(
        sheet="Sheet1", 
        index=None, 
    )
    
    print(f"Successfully loaded {len(df_sheet)} rows of agent data.")
    # print(df_sheet.tail())
except Exception as e:
    print(f"Error: {e}")

Successfully loaded 428 rows of agent data.


In [ ]:
from gspread_pandas import Spread, conf
import pandas as pd

# 1. Setup Connection
config_path = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
c = conf.get_config(file_name=config_path)

# Google Sheet URL
list_sheet = "https://docs.google.com/spreadsheets/d/1ZexyNYrwg1p7EG6q2S1Fv4XfliDKkWR0T1YeYXo84gU/edit?gid=362506180"

# 2. Connect to the sheet
spread = Spread(list_sheet, config=c)

df1 = spread.sheet_to_df(header_rows=5, index=None)
df2 = pd.read_excel(r'C:\Users\USER\Downloads\Emarath_global\Output_Data\agent_mismatch_report.xlsx')

# 5. Clean data for matching
df1['MOBILE'] = df1['MOBILE'].astype(str).str.strip()
df2['MOBILE'] = df2['MOBILE'].astype(str).str.strip()


agent_map = df2.set_index('MOBILE')['AGENT_NAME_Report2'].to_dict()
df1['remarks'] = df1['MOBILE'].map(agent_map).fillna(df1['remarks'])

spread.df_to_sheet(df1, index=False, sheet='Sheet1', start='A6')

print("Update complete! Data pushed to 'remarks' skipping the date rows.")

Update complete! Data pushed to 'remarks' skipping the date rows.
